# User Third Transaction

## Problem Statement

For every user with at least three transactions, find their third transaction in chronological order.

Return the user identifier, spend amount, and transaction date associated with the third transaction.

Users with fewer than three transactions should not appear in the result.

## Input Table

### ttq_transaction_third

| Column Name | Data Type |
|------------|-----------|
| user_id | INT |
| spend | DECIMAL |
| transaction_date | TIMESTAMP |

## Requirements

- Consider transactions independently for each user.
- Order transactions chronologically using `transaction_date`.
- Identify the third transaction for each eligible user.
- Exclude users with fewer than three transactions.
- Transaction dates are unique within each user.
- Return results matching the required output schema and order.

## Output Columns

| Column Name |
|------------|
| user_id |
| spend |
| transaction_date |

## Sample Input

### ttq_transaction_third

| spend | user_id | transaction_date |
|--------|---------|------------------|
| 100.5 | 111 | 2022-01-08 |
| 55 | 111 | 2022-01-10 |
| 36 | 121 | 2022-01-18 |
| 24.99 | 145 | 2022-01-26 |
| 89.6 | 111 | 2022-02-05 |

## Sample Output

| user_id | spend | transaction_date |
|---------|-------|------------------|
| 111 | 89.6 | 2022-02-05 |

## Expected Output Schema

| Column Name | Data Type |
|------------|-----------|
| user_id | INT |
| spend | DECIMAL |
| transaction_date | TIMESTAMP |

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql import Window

ttq_transaction_third_schema = StructType([
    StructField("spend", DoubleType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("transaction_date", TimestampType(), True)
])

ttq_transaction_third_data = [
    (100.5, 111, "2022-01-08 00:00:00"),
    (55.0, 111, "2022-01-10 00:00:00"),
    (36.0, 121, "2022-01-18 00:00:00"),
    (24.99, 145, "2022-01-26 00:00:00"),
    (89.6, 111, "2022-02-05 00:00:00")
]

ttq_transaction_third_df = spark.createDataFrame(
    ttq_transaction_third_data,
    ["spend", "user_id", "transaction_date"]
).withColumn(
    "transaction_date",
    to_date(col("transaction_date"))
)

In [0]:
result_df = (
    ttq_transaction_third_df.withColumn(
        "rank",
        dense_rank().over(
            Window.partitionBy("user_id").orderBy(col("transaction_date"))
        ),
    )
    .filter(col("rank") == 3)
    .select("user_id", "spend", "transaction_date")
)
display(result_df)